# Session 09: LangGraph Platform

## Deploying the Stone Ridge Investment Assistant with LangGraph Platform

### Learning Objectives

- **Understand LangGraph Platform architecture** — how `langgraph.json`, `app/agent.py`, and the CLI work together
- **Walk through the consolidated agent** — understand the graph structure with guardrails, helpfulness evaluation, and investment-domain tools
- **Deploy locally with `langgraph dev`** — run the investment assistant as a local API server
- **Test via the LangGraph SDK** — interact with the deployed agent programmatically

### Overview

In this notebook we explore the **`app/agent.py`** file — a consolidated LangGraph agent that combines:
- **Anthropic Claude** as the LLM (Claude Sonnet for main agent, Claude Haiku for evaluation)
- **RAG** over the Stone Ridge 2025 Investor Letter
- **X/Twitter tools** for market sentiment analysis
- **Tavily + Arxiv** for web and academic search
- **Input/output guardrails** for production safety
- **Helpfulness evaluation** loop for response quality

The agent is deployed via **LangGraph Platform**, which provides:
- A REST API for invocation
- Built-in streaming support
- Thread-based memory management
- LangSmith integration for observability

---

## Task 1: Understand the Agent Architecture

Let's start by examining the key components of our consolidated agent.

### Graph Architecture

```
START \u2192 input_guardrail \u2192[pass]\u2192 agent \u2192[tools?]\u2192 action \u2192 agent (loop)
                         [fail]\u2192 END        [no tools]\u2192 output_guardrail \u2192[pass]\u2192 helpfulness \u2192[Y]\u2192 END
                                                                          [fail]\u2192 agent (retry)   [N]\u2192 agent
```

### Key Components

| Component | Description | Model |
|-----------|-------------|-------|
| Input Guardrail | Topic restriction, jailbreak detection, PII protection | Guardrails AI |
| Agent | Main reasoning + tool calling | Claude Sonnet |
| Action | Tool execution (Tavily, Arxiv, RAG, X/Twitter) | N/A |
| Output Guardrail | PII protection, profanity filter | Guardrails AI |
| Helpfulness | Response quality evaluation | Claude Haiku |

In [2]:
import os
import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API Key:")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key (for embeddings):")

# Optional
if not os.environ.get("TAVILY_API_KEY"):
    tavily = getpass.getpass("Tavily API Key (optional \u2014 Enter to skip):")
    if tavily.strip():
        os.environ["TAVILY_API_KEY"] = tavily

### Examining the Agent Code

Let's look at the key sections of `app/agent.py`:

In [2]:
# View the agent module structure
with open("app/agent.py") as f:
    lines = f.readlines()

# Print section headers
for i, line in enumerate(lines, 1):
    if line.startswith("# ----") or line.startswith("def ") or line.startswith("class "):
        print(f"{i:4d}: {line.rstrip()}")

  40: # ---------------------------------------------------------------------------
  42: # ---------------------------------------------------------------------------
  77: # ---------------------------------------------------------------------------
  79: # ---------------------------------------------------------------------------
  82: class AgentState(TypedDict):
  88: # ---------------------------------------------------------------------------
  90: # ---------------------------------------------------------------------------
  93: def get_chat_model(
 107: # ---------------------------------------------------------------------------
 109: # ---------------------------------------------------------------------------
 112: class CacheBackedEmbeddings:
 135: def setup_llm_cache(cache_type: str = "memory", cache_path: str | None = None):
 149: # ---------------------------------------------------------------------------
 151: # ------------------------------------------------------

---

## Task 2: Import and Compile the Graph Locally

Before deploying to LangGraph Platform, let's import and test the graph directly.

In [3]:
from app.agent import (
    graph,
    build_graph,
    get_chat_model,
    get_tool_belt,
    SYSTEM_PROMPT,
    DEFAULT_MODEL,
    EVAL_MODEL,
)

print(f"Graph compiled successfully!")
print(f"Default model: {DEFAULT_MODEL}")
print(f"Eval model: {EVAL_MODEL}")
print(f"\nTools available:")
for t in get_tool_belt():
    print(f"  - {t.name}")

/Users/jaden.lee/code/aimakerspace/09_Production_and_MCP/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Graph compiled successfully!
Default model: claude-sonnet-4-20250514
Eval model: claude-haiku-4-5-20251001

Tools available:
  - arxiv
  - retrieve_information
  - search_recent_posts
  - get_user_posts
  - get_post_by_id


In [ ]:
import os
import certifi

# Create a combined cert bundle with Zscaler for corporate network
zscaler_cert = "/Users/jaden.lee/ZscalerRootCertificate-2048-SHA256-Feb2025.crt"
combined_cert = "/Users/jaden.lee/combined_certs.pem"

with open(combined_cert, "w") as outfile:
    with open(certifi.where(), "r") as certifi_file:
        outfile.write(certifi_file.read())
    with open(zscaler_cert, "r") as zscaler_file:
        outfile.write(zscaler_file.read())

os.environ['REQUESTS_CA_BUNDLE'] = combined_cert
os.environ['SSL_CERT_FILE'] = combined_cert
os.environ['CURL_CA_BUNDLE'] = combined_cert

In [4]:
# Visualize the graph
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

---

## Task 3: Test the Graph Locally

Let's invoke the graph directly with some investment-related queries.

In [ ]:
from langchain_core.messages import HumanMessage

# Test with an investment question
result = graph.invoke(
    {"messages": [HumanMessage(content="What is Stone Ridge's investment philosophy regarding reinsurance?")]}
)

# Print the final response (skip HELPFULNESS markers)
for msg in reversed(result["messages"]):
    if hasattr(msg, "content") and not msg.content.startswith("HELPFULNESS:"):
        print(msg.content)
        break

In [ ]:
# Test the guardrails with an off-topic query
result = graph.invoke(
    {"messages": [HumanMessage(content="What medicine should I take for a headache?")]}
)

for msg in reversed(result["messages"]):
    if hasattr(msg, "content") and msg.content:
        print(msg.content)
        break

In [ ]:
# Test with a query that should use the RAG tool
result = graph.invoke(
    {"messages": [HumanMessage(
        content="According to the Stone Ridge 2025 Investor Letter, how does the firm think about bitcoin allocation?"
    )]}
)

for msg in reversed(result["messages"]):
    if hasattr(msg, "content") and not msg.content.startswith("HELPFULNESS:"):
        print(msg.content)
        break

---

## Task 4: Deploy with LangGraph Platform

Now let's deploy the agent using LangGraph Platform. The configuration is defined in `langgraph.json`:

```json
{
  "version": 1,
  "dependencies": ["."],
  "env": ".env",
  "python_version": "3.13",
  "graphs": {
    "investment_assistant": "app.agent:graph"
  }
}
```

### Starting the Server

In a terminal, run:

```bash
cd 09_Production_and_MCP
uv run langgraph dev
```

This will:
1. Install dependencies from `pyproject.toml`
2. Load environment variables from `.env`
3. Start a local API server at `http://localhost:2024`
4. Open the LangGraph Studio UI for visual debugging

---

## Task 5: Test via LangGraph SDK

Once the server is running, we can interact with it using the **LangGraph SDK**. This is how production clients would call the agent.

In [ ]:
from langgraph_sdk import get_sync_client

# Connect to the local LangGraph Platform server
client = get_sync_client(url="http://localhost:2024")

print("Connected to LangGraph Platform!")

# List available assistants
assistants = client.assistants.search()
for a in assistants:
    print(f"  - {a['assistant_id']}: {a.get('name', 'unnamed')}")

In [ ]:
# Stream a response from the deployed agent
for chunk in client.runs.stream(
    None,  # Threadless run
    "investment_assistant",
    input={
        "messages": [
            {
                "role": "human",
                "content": "What is Stone Ridge's view on reinsurance as an asset class?",
            }
        ]
    },
    stream_mode="updates",
):
    print(f"Event: {chunk.event}")
    if chunk.data:
        # Print the last message content if available
        messages = chunk.data.get("messages", [])
        for msg in messages:
            content = msg.get("content", "")
            if content and not content.startswith("HELPFULNESS:"):
                print(f"  {content[:200]}..." if len(content) > 200 else f"  {content}")
    print()

In [ ]:
# Test with a thread for multi-turn conversation
thread = client.threads.create()
print(f"Thread created: {thread['thread_id']}")

# First message
for chunk in client.runs.stream(
    thread["thread_id"],
    "investment_assistant",
    input={"messages": [{"role": "human", "content": "Tell me about Stone Ridge's bitcoin strategy."}]},
    stream_mode="updates",
):
    if chunk.event == "updates" and chunk.data:
        messages = chunk.data.get("messages", [])
        for msg in messages:
            content = msg.get("content", "")
            if content and not content.startswith("HELPFULNESS:"):
                print(content[:500])

In [ ]:
# Follow-up question (same thread = agent remembers context)
for chunk in client.runs.stream(
    thread["thread_id"],
    "investment_assistant",
    input={"messages": [{"role": "human", "content": "How does that compare to their reinsurance approach?"}]},
    stream_mode="updates",
):
    if chunk.event == "updates" and chunk.data:
        messages = chunk.data.get("messages", [])
        for msg in messages:
            content = msg.get("content", "")
            if content and not content.startswith("HELPFULNESS:"):
                print(content[:500])

### ❓ Question #1

What are the benefits of deploying an agent through LangGraph Platform versus running it directly as a Python script? Consider: scalability, observability, memory management, and client access.

##### Answer

- **Scalability**: LangGraph Platform wraps the agent in a REST API server, so multiple clients can invoke it concurrently without each needing their own Python process. In production, the platform can be deployed behind a load balancer and scaled horizontally, whereas a raw Python script is a single process bound to one machine.

- **Observability**: The platform integrates natively with LangSmith, providing automatic tracing of every node execution, tool call, and LLM invocation. With a plain script, you'd need to manually instrument logging and tracing yourself. The built-in LangGraph Studio UI also lets you visually step through graph execution for debugging.

- **Memory management**: LangGraph Platform provides thread-based persistence out of the box. Each thread maintains its own conversation history across requests (as demonstrated in Tasks 5 cells 17–18, where a follow-up question reuses the same thread and the agent remembers prior context). With a raw script, you'd need to build your own state storage, serialization, and session management.

- **Client access**: The platform exposes a standardized REST API and streaming protocol, enabling any language or framework to interact with the agent via the LangGraph SDK or plain HTTP. A Python script would require building your own API layer (e.g., FastAPI wrapper), handling serialization, streaming SSE events, and error handling—all of which LangGraph Platform provides as built-in infrastructure.

### ❓ Question #2

How does the helpfulness evaluation loop affect latency and cost? When would you enable or disable it in production?

##### Answer

**Latency impact**: The helpfulness node adds at least one extra LLM call (using Claude Haiku) after every agent response. If the evaluator returns "N" (unhelpful), the response loops back to the agent node for another full LLM call, potentially triggering additional tool calls and another helpfulness evaluation. In the worst case, this loop can repeat multiple times until the response is deemed helpful or the message count exceeds 10 (the safety cap at `agent.py:523`). This can easily double or triple end-to-end latency.

**Cost impact**: Each helpfulness evaluation consumes tokens for both the prompt (initial query + full response) and the completion. If the loop triggers a retry, you pay for an entirely new agent invocation plus another evaluation. Using Haiku for evaluation keeps per-call cost low (~10-20x cheaper than Sonnet), but the multiplicative effect of retries can still add up significantly at scale.

**When to enable**:
- High-stakes use cases where response quality matters more than speed (e.g., investor-facing reports, compliance-sensitive answers)
- Async/batch workflows where latency is not user-facing
- During development and testing to understand baseline quality

**When to disable**:
- Real-time or interactive chat where users expect sub-second responses
- High-throughput scenarios where cost per request must be minimized
- When the agent has been validated to produce consistently high-quality responses without the loop (i.e., the retry rate is negligibly low, making the extra evaluation call pure overhead)

### \ud83c\udfd7\ufe0f Activity #1

Modify the agent to add a new graph variant:

1. Create a "simple" variant in `app/agent.py` that skips guardrails and helpfulness (just agent + action nodes)
2. Register it in `langgraph.json` as `"simple_assistant"`
3. Restart `langgraph dev` and test both variants via the SDK
4. Compare response times and quality

In [4]:
import time
from langchain_core.messages import HumanMessage
from app.agent import graph, simple_graph

query = "What is Stone Ridge's investment philosophy regarding reinsurance?"
msg = {"messages": [HumanMessage(content=query)]}

# --- Full graph (guardrails + helpfulness) ---
t0 = time.time()
full_result = graph.invoke(msg)
full_time = time.time() - t0

full_response = None
for m in reversed(full_result["messages"]):
    if hasattr(m, "content") and not m.content.startswith("HELPFULNESS:"):
        full_response = m.content
        break

# --- Simple graph (agent + tools only) ---
t0 = time.time()
simple_result = simple_graph.invoke(msg)
simple_time = time.time() - t0

simple_response = None
for m in reversed(simple_result["messages"]):
    if hasattr(m, "content"):
        simple_response = m.content
        break

# --- Compare ---
print("=" * 60)
print("FULL GRAPH (guardrails + helpfulness)")
print(f"  Time: {full_time:.2f}s")
print(f"  Response: {full_response[:300]}...")
print()
print("=" * 60)
print("SIMPLE GRAPH (agent + tools only)")
print(f"  Time: {simple_time:.2f}s")
print(f"  Response: {simple_response[:300]}...")
print()
print("=" * 60)
print(f"Speed difference: {full_time - simple_time:+.2f}s (simple is {full_time / simple_time:.1f}x faster)")

Guardrails not available — running without input guards: cannot import name 'DetectJailbreak' from 'guardrails.hub' (/Users/jaden.lee/code/aimakerspace/09_Production_and_MCP/.venv/lib/python3.14/site-packages/guardrails/hub/__init__.py)
Guardrails not available — running without output guards: cannot import name 'GuardrailsPII' from 'guardrails.hub' (/Users/jaden.lee/code/aimakerspace/09_Production_and_MCP/.venv/lib/python3.14/site-packages/guardrails/hub/__init__.py)


FULL GRAPH (guardrails + helpfulness)
  Time: 48.53s
  Response: Based on Stone Ridge's 2025 Investor Letter, here is their investment philosophy regarding reinsurance:

## Core Philosophy: Partnership Over Competition

Stone Ridge's reinsurance strategy through **Longtail Re** (founded in 2020) is built on a fundamental philosophy of **partnership rather than co...

SIMPLE GRAPH (agent + tools only)
  Time: 25.39s
  Response: Based on Stone Ridge's 2025 Investor Letter, here's their investment philosophy regarding reinsurance:

## Core Philosophy: Partner, Don't Compete

Stone Ridge's reinsurance strategy through **Longtail Re** is built on two fundamental principles:

1. **Partner with the world's best underwriters** to...

Speed difference: +23.14s (simple is 1.9x faster)


---

## Summary

In this notebook we covered:

- **Agent Architecture**: Walked through the consolidated `app/agent.py` with guardrails, tools, and helpfulness evaluation
- **Local Testing**: Imported and tested the graph directly in Python
- **LangGraph Platform Deployment**: Used `langgraph dev` to deploy as a local API server
- **SDK Client**: Tested the deployed agent with streaming and multi-turn threads

### Key Takeaways

1. **LangGraph Platform = deployment made simple** \u2014 `langgraph.json` + `app/agent.py` is all you need
2. **The SDK provides production-grade access** \u2014 threads, streaming, and async are built in
3. **Guardrails add safety but also latency** \u2014 consider the tradeoff per use case
4. **The helpfulness loop improves quality** \u2014 but costs an extra LLM call per response